# Deep Learning MCQ Solver – Kaggle Test Notebook

**How to use this notebook on Kaggle:**

1. In the right panel → **Add data** → search for `Qwen2-VL-7B-Instruct` model → add it
2. Also add your test dataset (test.csv + images/) as input
3. Set accelerator to **GPU T4 x2**
4. Run all cells
5. Download `submission.csv` from the output

---
**Note:** This notebook tests on Kaggle (T4x2). Actual grading runs on L40s via `inference.py`.

In [ ]:
# ── Install / verify packages ────────────────────────────────
import subprocess, sys

packages = [
    'transformers>=4.45.0',
    'accelerate',
    'bitsandbytes',
    'qwen-vl-utils',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages ready')

In [ ]:
import os
import re
import time
import pandas as pd
from PIL import Image
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# ── Locate model ─────────────────────────────────────────────
# After adding Qwen2-VL-7B-Instruct model on Kaggle,
# it usually appears at one of these paths:
KAGGLE_MODEL_PATHS = [
    '/kaggle/input/qwen2-vl-7b-instruct',
    '/kaggle/input/qwen2-vl/transformers/qwen2-vl-7b-instruct/1',
    '/kaggle/input/qwen2vl7binstruct/transformers/default/1',
]

model_path = None
for p in KAGGLE_MODEL_PATHS:
    if os.path.isdir(p) and any(
        f.endswith('.safetensors') or f.endswith('.bin')
        for f in os.listdir(p)
    ):
        model_path = p
        break

if model_path is None:
    # List available Kaggle inputs to help debug
    print('Available Kaggle inputs:')
    for d in os.listdir('/kaggle/input'):
        print(' ', d)
    raise FileNotFoundError(
        'Model not found! Add Qwen2-VL-7B-Instruct as a Kaggle model input.'
    )

print(f'Model found at: {model_path}')

In [ ]:
# ── Load model (4-bit for T4 16GB) ───────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Loading model...')
t0 = time.time()
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained(model_path)
model.eval()
print(f'Model loaded in {time.time()-t0:.1f}s')

In [ ]:
# ── Answer function ───────────────────────────────────────────
USER_PROMPT = """Carefully examine the deep learning multiple-choice question in this image.

Follow these steps:
1. Read the question and understand exactly what is being asked.
2. Review each option (A, B, C, D) carefully.
3. Apply relevant deep learning concepts to reason through the answer.
4. Think step by step to identify the correct option.

At the very end of your response, you MUST write your final answer in EXACTLY this format:
FINAL_ANSWER: [number]

Where:
1  →  Option A is correct
2  →  Option B is correct
3  →  Option C is correct
4  →  Option D is correct
5  →  Cannot determine"""

def parse_answer(text):
    match = re.search(r'FINAL_ANSWER\s*[:=]\s*([1-5])', text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    letter_match = re.search(
        r'(?:answer|correct option)\s*(?:is|:)?\s*[(\[]?\s*([A-D])\s*[)\]]?',
        text, re.IGNORECASE
    )
    if letter_match:
        return 'ABCD'.index(letter_match.group(1).upper()) + 1
    digits = re.findall(r'\b([1-4])\b', text)
    if digits:
        return int(digits[-1])
    return 5

def answer_mcq(image_path):
    image = Image.open(image_path).convert('RGB')
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': USER_PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors='pt'
    )
    device = next(model.parameters()).device
    inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}

    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=600, do_sample=False)

    in_len = inputs['input_ids'].shape[1]
    out = processor.decode(gen_ids[0][in_len:], skip_special_tokens=True).strip()
    print(f'  Output (last 200 chars): ...{out[-200:]}')
    return parse_answer(out)

print('Answer function ready')

In [ ]:
# ── Set your test directory ───────────────────────────────────
# Change this to wherever your test data is on Kaggle
TEST_DIR = '/kaggle/input/your-test-dataset'   # <-- UPDATE THIS

test_df = pd.read_csv(os.path.join(TEST_DIR, 'test.csv'))
images_dir = os.path.join(TEST_DIR, 'images')
print(f'Questions: {len(test_df)}')
test_df.head()

In [ ]:
# ── Run inference ─────────────────────────────────────────────
results = []
total = len(test_df)
start = time.time()

for idx, row in test_df.iterrows():
    image_id   = row.get('image_id', row['image_name'])
    image_name = row['image_name']

    for ext in ['.png', '', '.jpg', '.jpeg']:
        img_path = os.path.join(images_dir, f'{image_name}{ext}')
        if os.path.exists(img_path):
            break
    else:
        img_path = None

    print(f'\n[{idx+1}/{total}] {image_name}')
    if img_path is None:
        answer = 5
    else:
        t0 = time.time()
        answer = answer_mcq(img_path)
        print(f'  → Answer: {answer}  ({time.time()-t0:.1f}s)')

    results.append({'id': image_id, 'image_name': image_name, 'option': answer})

print(f'\nTotal time: {(time.time()-start)/60:.1f} min')

In [ ]:
# ── Save submission ───────────────────────────────────────────
submission = pd.DataFrame(results)
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
submission